# hls4ml on PYNQ-Z2 — on-board inference

Loads any of the project's bitstreams, runs the full `X_test.npy` through, reports accuracy + timing.

**Usage:**
1. SCP a `<config>.bit`, `<config>.hwh`, `X_test.npy`, `y_test.npy` into the **same directory** as this notebook on the PYNQ board.
2. Edit `MODEL_DIR` in the config cell below if files are elsewhere.
3. Run all cells.

**Supported bitstreams** (all built from `hls4ml VivadoAccelerator`, board = pynq-z2):
- ECG v1 — `uniform_8bit` or `low_mixed` (16 features)
- ECG v2 — `uniform_8bit` or `low_mixed` (64 features)
- NID — `uniform_8bit` or `low_mixed` (41 features)

In [ ]:
# ===== CONFIG — edit these =====
import os

MODEL_DIR    = '.'                  # directory holding .bit/.hwh/X_test.npy/y_test.npy
CONFIG       = 'uniform_8bit'       # the prefix shared by <CONFIG>.bit and <CONFIG>.hwh
INPUT_DIM    = 16                   # 16 = ECG v1, 64 = ECG v2, 41 = NID
OUTPUT_DIM   = 1                    # all our models are binary classifiers
DECISION_THR = 0.5                  # sigmoid threshold for binary classification
MAX_SAMPLES  = None                 # None = use whole X_test, or set e.g. 1000 for a quick smoke test

BIT  = os.path.join(MODEL_DIR, f'{CONFIG}.bit')
HWH  = os.path.join(MODEL_DIR, f'{CONFIG}.hwh')
XTEST = os.path.join(MODEL_DIR, 'X_test.npy')
YTEST = os.path.join(MODEL_DIR, 'y_test.npy')

for p in [BIT, HWH, XTEST, YTEST]:
    assert os.path.exists(p), f'Missing: {p}'
print('Files OK:', BIT, HWH, XTEST, YTEST, sep='\n  ')

In [ ]:
# ===== Imports =====
import time
import numpy as np
from pynq import Overlay, allocate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# ===== Driver class (matches hls4ml's auto-generated axi_stream_driver.py) =====
class NeuralNetworkOverlay(Overlay):
    """hls4ml VivadoAccelerator overlay wrapper.

    The HLS-generated AXI-Stream interface uses float32 on both ends; the
    float-to-ap_fixed cast happens inside the HLS wrapper. So Python just
    sends/receives np.float32 arrays through DMA — no manual fixed-point
    encoding needed here.
    """
    def __init__(self, bitfile, x_shape, y_shape, dtype=np.float32):
        super().__init__(bitfile, download=True, ignore_version=True)
        self.sendchannel = self.hier_0.axi_dma_0.sendchannel
        self.recvchannel = self.hier_0.axi_dma_0.recvchannel
        self.input_buffer  = allocate(shape=x_shape, dtype=dtype)
        self.output_buffer = allocate(shape=y_shape, dtype=dtype)

    def predict_one(self, x):
        """Single sample inference. x is 1-D, returns 1-D output."""
        self.input_buffer[:] = x.astype(np.float32)
        self.sendchannel.transfer(self.input_buffer)
        self.recvchannel.transfer(self.output_buffer)
        self.sendchannel.wait()
        self.recvchannel.wait()
        return np.array(self.output_buffer, copy=True)

    def close(self):
        self.input_buffer.freebuffer()
        self.output_buffer.freebuffer()

In [ ]:
# ===== Load test data and bitstream =====
X_test = np.load(XTEST)
y_test = np.load(YTEST)

if MAX_SAMPLES is not None:
    X_test = X_test[:MAX_SAMPLES]
    y_test = y_test[:MAX_SAMPLES]

print(f'X_test shape: {X_test.shape}, dtype: {X_test.dtype}')
print(f'y_test shape: {y_test.shape}, label distribution:',
      dict(zip(*np.unique(y_test, return_counts=True))))
assert X_test.shape[1] == INPUT_DIM, (
    f'INPUT_DIM mismatch — config says {INPUT_DIM} but data has {X_test.shape[1]}'
)

print(f'\nLoading overlay: {BIT}')
ol = NeuralNetworkOverlay(BIT, x_shape=(INPUT_DIM,), y_shape=(OUTPUT_DIM,))
print('Overlay loaded.')

In [ ]:
# ===== Smoke test — first sample only =====
print('First sample input:', X_test[0])
print('First sample label:', y_test[0])
y0 = ol.predict_one(X_test[0])
print('FPGA raw output:', y0)
print('FPGA decision:', int(y0[0] > DECISION_THR))
print('Match?', int(y0[0] > DECISION_THR) == int(y_test[0]))

In [ ]:
# ===== Full sweep — accuracy + per-sample latency =====
N = len(X_test)
preds = np.zeros(N, dtype=np.int8)
raw_outputs = np.zeros(N, dtype=np.float32)

t0 = time.perf_counter()
for i in range(N):
    y = ol.predict_one(X_test[i])
    raw_outputs[i] = y[0]
    preds[i] = int(y[0] > DECISION_THR)
    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{N}')
elapsed = time.perf_counter() - t0

print(f'\n=== Results ===')
print(f'Samples           : {N}')
print(f'Wall time         : {elapsed*1000:.1f} ms')
print(f'Per-sample latency: {elapsed*1000/N:.2f} ms')
print(f'Throughput        : {N/elapsed:.0f} inferences/s')
print(f'Accuracy          : {accuracy_score(y_test, preds):.4f}')
print()
print('Confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y_test, preds))
print()
print(classification_report(y_test, preds, target_names=['class 0', 'class 1']))

In [ ]:
# ===== Cleanup =====
ol.close()
print('Buffers freed. To switch to another model: change CONFIG + INPUT_DIM in the first cell, then re-run.')